In [ ]:
!pip install sentence-transformers faiss-cpu google-generativeai torch_geometric -q

In [8]:
# Twitter — training/train_twitter.py (ego network + degree-tier labels)
# SNAP ego-Twitter raw files: copy Kaggle add-on data into PyG’s raw/ so nothing is re-downloaded.
import glob
import logging
import os
import shutil

import numpy as np
import pandas as pd
import torch
from torch_geometric.datasets import SNAPDataset
from torch_geometric.utils import to_undirected

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Writable root for SNAPDataset (must not be the read-only /kaggle/input path)
SNAP_ROOT = os.environ.get("SNAP_DATA_ROOT", "/kaggle/working/snap_twitter")
# Kaggle dataset add-on: *.circles, *.edges, *.egofeat, *.feat, *.featnames
TWITTER_SNAP_SOURCE = os.environ.get(
    "TWITTER_SNAP_SOURCE",
    "/kaggle/input/datasets/akshatsharma1011/twitter-files/twitter",
)

FEATURE_DIM = 128


def _ego_twitter_raw_dir() -> str:
    # PyG uses lower-case: SNAPDataset(name="ego-Twitter") -> .../ego-twitter/raw
    return os.path.join(SNAP_ROOT, "ego-twitter", "raw")


def prepare_snap_twitter_raw() -> bool:
    """Copy local SNAP ego-Twitter files into PyG’s raw_dir; clear processed cache to reprocess."""
    if not os.path.isdir(TWITTER_SNAP_SOURCE):
        logger.info("Optional twitter-files folder not found: %s (will use CSV or download).", TWITTER_SNAP_SOURCE)
        return False
    raw_dir = _ego_twitter_raw_dir()
    os.makedirs(raw_dir, exist_ok=True)
    n = 0
    for path in glob.glob(os.path.join(TWITTER_SNAP_SOURCE, "*")):
        if os.path.isfile(path):
            shutil.copy2(path, os.path.join(raw_dir, os.path.basename(path)))
            n += 1
    processed_pt = os.path.join(SNAP_ROOT, "ego-twitter", "processed", "data.pt")
    if os.path.isfile(processed_pt):
        os.remove(processed_pt)
    logger.info("Copied %d files from %s -> %s", n, TWITTER_SNAP_SOURCE, raw_dir)
    return n > 0


def _raw_dir_has_files() -> bool:
    d = _ego_twitter_raw_dir()
    if not os.path.isdir(d):
        return False
    return any(os.path.isfile(os.path.join(d, f)) for f in os.listdir(d))


def assign_community_labels(edge_index, num_nodes):
    degrees = torch.zeros(num_nodes)
    degrees.scatter_add_(0, edge_index[0], torch.ones(edge_index.size(1)))
    degrees.scatter_add_(0, edge_index[1], torch.ones(edge_index.size(1)))
    q33 = degrees.quantile(0.33).item()
    q66 = degrees.quantile(0.66).item()
    q90 = degrees.quantile(0.90).item()
    labels = torch.zeros(num_nodes, dtype=torch.long)
    labels[degrees > q33] = 1
    labels[degrees > q66] = 2
    labels[degrees > q90] = 3
    return labels


def build_structural_features(edge_index, num_nodes, feature_dim):
    deg = torch.zeros(num_nodes)
    deg.scatter_add_(0, edge_index[0], torch.ones(edge_index.size(1)))
    deg.scatter_add_(0, edge_index[1], torch.ones(edge_index.size(1)))
    deg_norm = (deg - deg.mean()) / (deg.std() + 1e-8)
    rnd = torch.randn(num_nodes, feature_dim - 1) * 0.1
    return torch.cat([deg_norm.unsqueeze(1), rnd], dim=1)


def load_twitter_ego():
    copied = prepare_snap_twitter_raw()

    kaggle_path = "/kaggle/input/twitter-social-circles/twitter_edges.csv"
    if (not _raw_dir_has_files()) and os.path.exists(kaggle_path):
        df = pd.read_csv(kaggle_path)
        src = torch.tensor(df.iloc[:, 0].values, dtype=torch.long)
        dst = torch.tensor(df.iloc[:, 1].values, dtype=torch.long)
        edge_index = to_undirected(torch.stack([src, dst]))
        num_nodes = int(edge_index.max().item()) + 1
        y = assign_community_labels(edge_index, num_nodes)
        return edge_index, y, num_nodes

    logger.info(
        "Loading from SNAPDataset (raw under %s; PyG skips download when raw/ is non-empty).",
        _ego_twitter_raw_dir(),
    )
    dataset = SNAPDataset(root=SNAP_ROOT, name="ego-Twitter", force_reload=copied)

    all_s, all_d = [], []
    off = 0
    for d in dataset:
        if d.edge_index.size(1) == 0:
            continue
        all_s.append(d.edge_index[0] + off)
        all_d.append(d.edge_index[1] + off)
        off += d.num_nodes
    edge_index = to_undirected(torch.stack([torch.cat(all_s), torch.cat(all_d)]))
    num_nodes = off
    y = assign_community_labels(edge_index, num_nodes)
    return edge_index, y, num_nodes


edge_index, y_t, n_nodes = load_twitter_ego()
Y = y_t.numpy()
nodes = list(range(n_nodes))
print("Total nodes:", len(nodes))


INFO:__main__:Copied 4865 files from /kaggle/input/datasets/akshatsharma1011/twitter-files/twitter -> /kaggle/working/snap_twitter/ego-twitter/raw
INFO:__main__:Loading from SNAPDataset (raw under /kaggle/working/snap_twitter/ego-twitter/raw; PyG skips download when raw/ is non-empty).
Processing...
  0%|          | 0/973 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch_geometric/datasets/snap_dataset.py:72: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:49.)
  x = x.to_sparse_csr()
100%|██████████| 973/973 [02:56<00:00,  5.51it/s]
Done!
/usr/local/lib/python3.12/dist-packages/torch_geometric/data/in_memory_dataset.py:131: UserWarning: Weights only load failed. Please file an issue to make `torch.load(weights_only=True)` compatible in your case. Please use `t

Total nodes: 133857


In [9]:
documents = []
for node in nodes:
    doc = (
        f"Node ID: {node}\n"
        f"Twitter user in merged SNAP ego-Twitter networks; edges are follow relationships.\n"
        f"Activity tier / class label is not stated in this text.\n"
    )
    documents.append(doc)
print(documents[:2])


['Node ID: 0\nThis Twitter user is in an ego network; activity tier: active.\nEdges represent follow relationships.\n', 'Node ID: 1\nThis Twitter user is in an ego network; activity tier: active.\nEdges represent follow relationships.\n']


In [10]:
from sentence_transformers import SentenceTransformer
encoder = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = encoder.encode(documents, show_progress_bar=True)
print("Embedding shape:", embeddings.shape)

INFO:datasets:TensorFlow version 2.19.0 available.
INFO:datasets:JAX version 0.7.2 available.
INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cuda:0
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: all-MiniLM-L6-v2
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"
IN

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/tokenizer_config.json "HT

Batches:   0%|          | 0/4184 [00:00<?, ?it/s]

Embedding shape: (133857, 384)


In [11]:
import faiss
import numpy as np
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings).astype("float32"))
print("FAISS index ready")

INFO:faiss.loader:Loading faiss with AVX512 support.
INFO:faiss.loader:Successfully loaded faiss with AVX512 support.


FAISS index ready


In [ ]:
import google.generativeai as genai

genai.configure(api_key="your-gemini-api-key")

model = genai.GenerativeModel("gemini-2.5-flash")

print("Gemini ready")

Gemini ready


In [34]:
def retrieve(query, top_k=5):
    q_emb = encoder.encode([query])
    
    distances, indices = index.search(np.array(q_emb), top_k)
    
    return [documents[i] for i in indices[0]]

In [ ]:
def rag_query(query, top_k=5):
    context_docs = retrieve(query, top_k)
    
    context = "\n".join(context_docs)
    
    prompt = f"""
    You are a graph intelligence assistant.

    Context from Twitter graph dataset:
    {context}

    Question:
    {query}

    Answer clearly using the context.
    """
    
    response = model.generate_content(prompt)
    
    print("🔍 Query:", query)
    print("\n📄 Retrieved Context:\n")
    for doc in context_docs:
        print("-", doc.strip())
    
    print("\n🤖 Gemini Answer:\n")
    print(response.text)

In [36]:
rag_query("What activity tiers (roles) show up in this Twitter network?")


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

🔍 Query: What activity tiers (roles) show up in this Twitter network?

📄 Retrieved Context:

- Node ID: 13772
This Twitter user is in an ego network; activity tier: active.
Edges represent follow relationships.
- Node ID: 67038
This Twitter user is in an ego network; activity tier: active.
Edges represent follow relationships.
- Node ID: 20147
This Twitter user is in an ego network; activity tier: active.
Edges represent follow relationships.
- Node ID: 6707
This Twitter user is in an ego network; activity tier: active.
Edges represent follow relationships.
- Node ID: 52172
This Twitter user is in an ego network; activity tier: active.
Edges represent follow relationships.

🤖 Gemini Answer:

Based on the context provided, the only activity tier that shows up in this Twitter network is:

*   **active**


In [37]:
rag_query("Summarize what the retrieved user nodes have in common.")


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

🔍 Query: Summarize what the retrieved user nodes have in common.

📄 Retrieved Context:

- Node ID: 1977
This Twitter user is in an ego network; activity tier: regular.
Edges represent follow relationships.
- Node ID: 1939
This Twitter user is in an ego network; activity tier: regular.
Edges represent follow relationships.
- Node ID: 1943
This Twitter user is in an ego network; activity tier: lurker.
Edges represent follow relationships.
- Node ID: 1789
This Twitter user is in an ego network; activity tier: regular.
Edges represent follow relationships.
- Node ID: 19437
This Twitter user is in an ego network; activity tier: regular.
Edges represent follow relationships.

🤖 Gemini Answer:

All of the retrieved user nodes represent **Twitter users** who are part of an **ego network**. Additionally, for all these nodes, the **edges represent follow relationships**.


In [38]:
rag_query("Give examples of nodes described as active or influencer.")


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

🔍 Query: Give examples of nodes described as active or influencer.

📄 Retrieved Context:

- Node ID: 19777
This Twitter user is in an ego network; activity tier: influencer.
Edges represent follow relationships.
- Node ID: 19780
This Twitter user is in an ego network; activity tier: influencer.
Edges represent follow relationships.
- Node ID: 19952
This Twitter user is in an ego network; activity tier: influencer.
Edges represent follow relationships.
- Node ID: 18362
This Twitter user is in an ego network; activity tier: influencer.
Edges represent follow relationships.
- Node ID: 1926
This Twitter user is in an ego network; activity tier: influencer.
Edges represent follow relationships.

🤖 Gemini Answer:

Based on the context provided, here are examples of nodes described as an influencer:

*   **Node ID: 19777** (activity tier: influencer)
*   **Node ID: 19780** (activity tier: influencer)
*   **Node ID: 19952** (activity tier: influencer)
*   **Node ID: 18362** (activity tier: inf